In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb
from scipy import stats
from scipy.stats import wilcoxon
import time
import json
import warnings
import gc
import copy
import math
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

try:
    import einops
    from einops import rearrange
    SAINT_AVAILABLE = True
    print(f"einops {einops.__version__} loaded successfully")
except ImportError:
    SAINT_AVAILABLE = False
    print("WARNING: einops not found. Installing...")
    import subprocess
    subprocess.run(["pip", "install", "einops"], check=True)
    import einops
    from einops import rearrange
    SAINT_AVAILABLE = True
    print(f"einops installed and loaded successfully")

warnings.filterwarnings('ignore')

# --- Constants ---
K_FOLDS          = 5
CV_RANDOM_STATE  = 42
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SALES_FEATURES = 5
N_VIEW_FEATURES  = 5
N_TOTAL_FEATURES = N_SALES_FEATURES + N_VIEW_FEATURES  # 10

print(f"Device: {DEVICE}")
print(f"K Folds: {K_FOLDS}")
print(f"Feature dimensions: Sales={N_SALES_FEATURES}, "
      f"Engagement={N_VIEW_FEATURES}, Total={N_TOTAL_FEATURES}")

def set_all_seeds(seed):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def reset_gpu_stats():
    if DEVICE.type == 'cuda':
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

def get_peak_gpu_memory_mb():
    if DEVICE.type == 'cuda':
        return torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
    return 0.0


einops 0.8.2 loaded successfully
Device: cuda
K Folds: 5
Feature dimensions: Sales=5, Engagement=5, Total=10


In [3]:
class ImprovedDataProcessor:
    def __init__(self):
        self.scaler           = StandardScaler()
        self.sales_scaler     = StandardScaler()
        self.view_scaler      = StandardScaler()
        self.sales_scaler_xgb = StandardScaler()
        self.y_log_max        = None

    def load_full_dataset(self, excel_path='restaurant_merged_data.xlsx'):
        return pd.read_excel(excel_path, sheet_name='Merged_Data')

    def get_stratify_groups(self, data, test_size_for_rare_calc=0.1):
        category_counts    = data['sales_main_category'].value_counts()
        min_samples_needed = int(np.ceil(1 / test_size_for_rare_calc)) + 1
        rare_categories    = category_counts[
            category_counts < min_samples_needed].index.tolist()
        stratify_groups    = data['sales_main_category'].copy()
        if len(rare_categories) > 0:
            stratify_groups = stratify_groups.replace(
                rare_categories, 'RARE_COMBINED')
        return stratify_groups

    def prepare_features_fair(self, data_df, label_encoder, is_train=True):
        categories = (data_df['sales_main_category']
                      .fillna('Unknown').astype(str).str.strip())
        category_encoded = np.array([
            label_encoder.transform([cat])[0]
            if cat in label_encoder.classes_ else
            (label_encoder.transform(['Unknown'])[0]
             if 'Unknown' in label_encoder.classes_ else 0)
            for cat in categories
        ])
        target = data_df['net_sales_qty'].fillna(0).values

        net_sales_amount = data_df['net_sales_amount'].fillna(0).values
        sales_features   = [net_sales_amount, np.log1p(net_sales_amount)]
        for col in ('gross_sales_amount', 'cost', 'discount_amount'):
            if col in data_df.columns:
                sales_features.append(data_df[col].fillna(0).values)

        view_count    = data_df['view_count'].fillna(0).values
        view_features = [view_count, np.log1p(view_count)]
        for col in ('view_duration', 'avg_view_duration'):
            if col in data_df.columns:
                view_features.append(data_df[col].fillna(0).values)
        view_features.append((view_count > 0).astype(int))

        X_sales   = np.column_stack(sales_features)
        X_view    = np.column_stack(view_features)
        X_all     = np.column_stack([X_sales, X_view])
        view_mask = (view_count > 0)

        y     = np.maximum(target, 0.1)
        y_log = np.log1p(y)
        if is_train:
            self.y_log_max = y_log.max()
            if self.y_log_max == 0 or self.y_log_max is None:
                self.y_log_max = 1.0
        elif self.y_log_max is None:
            self.y_log_max = 1.0
        y_norm = y_log / self.y_log_max

        return X_sales, X_view, X_all, y_norm, y_log, \
               category_encoded, view_mask

    def prepare_features_xgb_native(self, data_df,
                                     label_encoder, is_train=True):
        categories = (data_df['sales_main_category']
                      .fillna('Unknown').astype(str).str.strip())
        category_encoded = np.array([
            label_encoder.transform([cat])[0]
            if cat in label_encoder.classes_ else
            (label_encoder.transform(['Unknown'])[0]
             if 'Unknown' in label_encoder.classes_ else 0)
            for cat in categories
        ])
        target = data_df['net_sales_qty'].fillna(0).values

        net_sales_amount = data_df['net_sales_amount'].fillna(0).values
        sales_features   = [net_sales_amount, np.log1p(net_sales_amount)]
        for col in ('gross_sales_amount', 'cost', 'discount_amount'):
            if col in data_df.columns:
                sales_features.append(data_df[col].fillna(0).values)
        X_sales = np.column_stack(sales_features)

        view_count_raw = data_df['view_count'].values
        view_mask      = (view_count_raw > 0)
        view_count_log = np.log1p(view_count_raw)

        if 'view_duration' in data_df.columns:
            view_duration = np.where(
                view_mask, data_df['view_duration'].values, np.nan)
        else:
            view_duration = np.where(view_mask, 0.0, np.nan)

        if 'avg_view_duration' in data_df.columns:
            avg_view_dur = np.where(
                view_mask,
                data_df['avg_view_duration'].fillna(0).values,
                np.nan)
        else:
            avg_view_dur = np.where(view_mask, 0.0, np.nan)

        view_indicator = view_mask.astype(float)

        X_view_native = np.column_stack([
            view_count_raw,
            view_count_log,
            view_duration,
            avg_view_dur,
            view_indicator
        ])
        X_all_native = np.column_stack([X_sales, X_view_native])

        y     = np.maximum(target, 0.1)
        y_log = np.log1p(y)
        if is_train:
            self.y_log_max = y_log.max()
            if self.y_log_max == 0 or self.y_log_max is None:
                self.y_log_max = 1.0
        elif self.y_log_max is None:
            self.y_log_max = 1.0

        return X_all_native, y_log, category_encoded, view_mask

_proc = ImprovedDataProcessor()
_df   = _proc.load_full_dataset()
_le   = LabelEncoder()
_le.fit(_df['sales_main_category'].fillna('Unknown').astype(str).str.strip())

_Xs, _Xv, _Xa, _yn, _yl, _cat, _mask = \
    _proc.prepare_features_fair(_df, _le, is_train=True)

_Xn, _yln, _catn, _maskn = \
    _proc.prepare_features_xgb_native(_df, _le, is_train=True)


In [4]:
class BaselineTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=2,
                 n_categories=18, dropout=0.1):
        super().__init__()
        self.category_embedding   = nn.Embedding(n_categories, 8)
        self.input_projection     = nn.Linear(input_dim, d_model)
        self.category_projection  = nn.Linear(8, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())

    def forward(self, x, categories, view_mask=None):
        x_proj   = self.input_projection(x)
        cat_emb  = self.category_embedding(categories)
        cat_proj = self.category_projection(cat_emb)
        sequence = torch.stack([x_proj, cat_proj], dim=1)
        out      = self.transformer(sequence)
        return self.output(out[:, 0, :]).squeeze(-1)


class DualStreamTransformer(nn.Module):
    def __init__(self, sales_dim, view_dim, d_model=32, nhead=2,
                 n_categories=18, dropout=0.1):
        super().__init__()
        self.category_embedding = nn.Embedding(n_categories, 8)
        self.sales_projection   = nn.Linear(sales_dim + 8, d_model)
        self.view_projection    = nn.Linear(view_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())

    def forward(self, x_sales, x_view, categories, view_mask):
        cat_emb          = self.category_embedding(categories)
        sales_with_cat   = torch.cat([x_sales, cat_emb], dim=1)
        sales_proj       = self.sales_projection(sales_with_cat)
        view_proj        = self.view_projection(x_view)
        view_mask_exp    = view_mask.unsqueeze(1).float()
        combined         = sales_proj + 0.5 * view_proj * view_mask_exp
        sequence         = combined.unsqueeze(1)
        out              = self.transformer(sequence)
        return self.output(out.squeeze(1)).squeeze(-1)


class AdaptiveWeightingTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=2,
                 n_categories=18, dropout=0.1, alpha=0.5):
        super().__init__()
        self.alpha                = alpha
        self.category_embedding   = nn.Embedding(n_categories, 8)
        self.feature_projection   = nn.Linear(input_dim, d_model)
        self.category_projection  = nn.Linear(8, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, 1)
        self.output = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())

    def forward(self, x, categories, view_mask):
        feat_proj        = self.feature_projection(x)
        mask_exp         = view_mask.unsqueeze(1).float()
        weighted         = feat_proj * (mask_exp + (1 - mask_exp) * self.alpha)
        cat_emb          = self.category_embedding(categories)
        cat_proj         = self.category_projection(cat_emb)
        sequence         = torch.stack([weighted, cat_proj], dim=1)
        out              = self.transformer(sequence)
        return self.output(out[:, 0, :]).squeeze(-1)


class GatedFusionTransformer(nn.Module):
    def __init__(self, sales_dim, full_dim, d_model=32, nhead=2,
                 num_layers=1, dropout=0.1, n_categories=18,
                 category_emb_dim=8):
        super().__init__()
        self.category_embedding = nn.Embedding(n_categories, category_emb_dim)
        self.sales_projection   = nn.Linear(
            sales_dim + category_emb_dim, d_model)
        sales_enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.sales_transformer  = nn.TransformerEncoder(sales_enc, num_layers)
        self.sales_output       = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())
        self.full_projection    = nn.Linear(
            full_dim + category_emb_dim, d_model)
        full_enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.full_transformer   = nn.TransformerEncoder(full_enc, num_layers)
        self.full_output        = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(16, 1), nn.ReLU())
        self.gate = nn.Sequential(
            nn.Linear(category_emb_dim + 1, 16), nn.ReLU(),
            nn.Linear(16, 2), nn.Softmax(dim=1))

    def forward(self, x_sales, x_all, categories, view_mask):
        cat_emb      = self.category_embedding(categories)
        sales_input  = torch.cat([x_sales, cat_emb], dim=1)
        sales_proj   = self.sales_projection(sales_input).unsqueeze(1)
        sales_out    = self.sales_transformer(sales_proj)
        sales_pred   = self.sales_output(sales_out.squeeze(1))
        full_input   = torch.cat([x_all, cat_emb], dim=1)
        full_proj    = self.full_projection(full_input).unsqueeze(1)
        full_out     = self.full_transformer(full_proj)
        full_pred    = self.full_output(full_out.squeeze(1))
        view_flag    = view_mask.float().unsqueeze(1)
        gate_input   = torch.cat([cat_emb, view_flag], dim=1)
        gate_weights = self.gate(gate_input)
        final_pred   = (gate_weights[:, 0:1] * sales_pred +
                        gate_weights[:, 1:2] * full_pred)
        return final_pred.squeeze(-1)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead    = nhead
        self.d_head   = d_model // nhead
        self.scale    = math.sqrt(self.d_head)
        self.q        = nn.Linear(d_model, d_model)
        self.k        = nn.Linear(d_model, d_model)
        self.v        = nn.Linear(d_model, d_model)
        self.out      = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(self, x):
        B, N, D  = x.shape
        q = self.q(x).reshape(B, N, self.nhead, self.d_head).transpose(1, 2)
        k = self.k(x).reshape(B, N, self.nhead, self.d_head).transpose(1, 2)
        v = self.v(x).reshape(B, N, self.nhead, self.d_head).transpose(1, 2)
        attn = torch.softmax(
            torch.matmul(q, k.transpose(-2, -1)) / self.scale, dim=-1)
        attn = self.dropout(attn)
        out  = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, D)
        return self.out(out)


class SAINTBlock(nn.Module):
    def __init__(self, n_features, d_model, nhead, dropout=0.1):
        super().__init__()
        self.col_norm1   = nn.LayerNorm(d_model)
        self.col_attn    = MultiHeadAttention(d_model, nhead, dropout)
        self.col_norm2   = nn.LayerNorm(d_model)
        self.col_ff      = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))

        row_dim          = n_features * d_model
        self.row_norm1   = nn.LayerNorm(row_dim)
        self.row_attn    = nn.MultiheadAttention(
            row_dim, 1, dropout=dropout, batch_first=True)
        self.row_norm2   = nn.LayerNorm(row_dim)
        self.row_ff      = nn.Sequential(
            nn.Linear(row_dim, row_dim * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(row_dim * 2, row_dim))

        self.n_features  = n_features
        self.d_model     = d_model

    def forward(self, x):
        B, N, D = x.shape

        x = x + self.col_attn(self.col_norm1(x))
        x = x + self.col_ff(self.col_norm2(x))

        x_flat   = x.reshape(B, N * D)
        x_flat   = x_flat.unsqueeze(1)
        x_norm   = self.row_norm1(x_flat)
        row_out, _ = self.row_attn(x_norm, x_norm, x_norm)
        x_flat   = x_flat + row_out
        x_flat   = x_flat + self.row_ff(
            self.row_norm2(x_flat))
        x        = x_flat.squeeze(1).reshape(B, N, D)

        return x


class SAINTModel(nn.Module):
    def __init__(self, n_cont_features=10, n_categories=18,
                 d_model=32, nhead=2, n_layers=2,
                 dropout=0.1, category_emb_dim=8):
        super().__init__()
        self.n_cont        = n_cont_features
        self.d_model       = d_model

        self.feat_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(n_cont_features)])

        self.cat_embedding   = nn.Embedding(n_categories, d_model)

        self.n_tokens        = n_cont_features + 1

        self.blocks          = nn.ModuleList([
            SAINTBlock(self.n_tokens, d_model, nhead, dropout)
            for _ in range(n_layers)])

        self.norm            = nn.LayerNorm(d_model)
        self.head            = nn.Sequential(
            nn.Linear(d_model * self.n_tokens, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.ReLU())

    def forward(self, x_cont, categories, view_mask=None):
        tokens = []
        for i, emb in enumerate(self.feat_embeddings):
            tokens.append(emb(x_cont[:, i:i+1]))

        cat_tok = self.cat_embedding(categories)
        tokens.append(cat_tok)

        x = torch.stack(tokens, dim=1)

        for block in self.blocks:
            x = block(x)

        x    = self.norm(x)
        x    = x.reshape(x.shape[0], -1)
        return self.head(x).squeeze(-1)

ConfigurableEnsembleTransformer = GatedFusionTransformer

print("Verifying model instantiation...")

_sales_dim = N_SALES_FEATURES
_view_dim  = N_VIEW_FEATURES
_input_dim = N_TOTAL_FEATURES
_n_cats    = 18

_m1 = BaselineTransformer(
    input_dim=_input_dim, n_categories=_n_cats)
_m2 = DualStreamTransformer(
    sales_dim=_sales_dim, view_dim=_view_dim, n_categories=_n_cats)
_m3 = AdaptiveWeightingTransformer(
    input_dim=_input_dim, n_categories=_n_cats)
_m4 = GatedFusionTransformer(
    sales_dim=_sales_dim, full_dim=_input_dim, n_categories=_n_cats)
_m5 = SAINTModel(
    n_cont_features=_input_dim, n_categories=_n_cats)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Baseline:            {count_params(_m1):,} parameters")
print(f"DualStream:          {count_params(_m2):,} parameters")
print(f"Adaptive:            {count_params(_m3):,} parameters")
print(f"ConfigurableEnsemble:{count_params(_m4):,} parameters")
print(f"SAINT:               {count_params(_m5):,} parameters")

_B = 4
_x = torch.randn(_B, _input_dim)
_c = torch.zeros(_B, dtype=torch.long)
_m = torch.ones(_B, dtype=torch.bool)
_xs = torch.randn(_B, _sales_dim)

with torch.no_grad():
    o1 = _m1(_x, _c)
    o2 = _m2(_xs, torch.randn(_B, _view_dim), _c, _m)
    o3 = _m3(_x, _c, _m)
    o4 = _m4(_xs, _x, _c, _m)
    o5 = _m5(_x, _c)

print(f"Forward pass shapes: "
      f"B={o1.shape}, DS={o2.shape}, "
      f"AW={o3.shape}, CE={o4.shape}, SAINT={o5.shape}")
print("All models output shape [4] confirmed.")

Verifying model instantiation...
Baseline:            9,873 parameters
DualStream:          9,873 parameters
Adaptive:            9,873 parameters
ConfigurableEnsemble:19,572 parameters
SAINT:               2,032,225 parameters
Forward pass shapes: B=torch.Size([4]), DS=torch.Size([4]), AW=torch.Size([4]), CE=torch.Size([4]), SAINT=torch.Size([4])
All models output shape [4] confirmed.


In [5]:
import os
Path('best_models').mkdir(exist_ok=True)


def get_metrics(y_pred_norm, y_true_log, processor):
    """Converts normalised predictions back to original scale and computes R2, MAE, MAPE."""
    pred_orig = np.expm1(y_pred_norm * processor.y_log_max)
    true_orig = np.expm1(y_true_log)
    pred_orig = np.maximum(pred_orig, 0.1)
    true_orig = np.maximum(true_orig, 0.1)
    return {
        'r2':   r2_score(true_orig, pred_orig),
        'mae':  mean_absolute_error(true_orig, pred_orig),
        'mape': np.median(
            np.abs((true_orig - pred_orig) /
                   np.maximum(true_orig, 1))) * 100
    }


def train_script1_model_run(model, processor, train_data, test_data,
                             method_name, seed, label_encoder,
                             lr=0.002, weight_decay=1e-5,
                             epochs=100, verbose=False):

    set_all_seeds(seed)
    reset_gpu_stats()
    best_loss = float('inf')

    try:
        if method_name == "Dual-Stream":
            X_sales_tr, X_view_tr, _, y_tr, y_log_tr, \
                cat_tr, mask_tr = \
                processor.prepare_features_fair(
                    train_data, label_encoder, is_train=True)
            X_sales_te, X_view_te, _, y_te, y_log_te, \
                cat_te, mask_te = \
                processor.prepare_features_fair(
                    test_data, label_encoder, is_train=False)
            processor.view_scaler = StandardScaler()
            X_sales_tr = torch.FloatTensor(
                processor.sales_scaler.fit_transform(X_sales_tr)).to(DEVICE)
            X_view_tr  = torch.FloatTensor(
                processor.view_scaler.fit_transform(X_view_tr)).to(DEVICE)
            X_sales_te = torch.FloatTensor(
                processor.sales_scaler.transform(X_sales_te)).to(DEVICE)
            X_view_te  = torch.FloatTensor(
                processor.view_scaler.transform(X_view_te)).to(DEVICE)
        else:
            _, _, X_all_tr, y_tr, y_log_tr, cat_tr, mask_tr = \
                processor.prepare_features_fair(
                    train_data, label_encoder, is_train=True)
            _, _, X_all_te, y_te, y_log_te, cat_te, mask_te = \
                processor.prepare_features_fair(
                    test_data, label_encoder, is_train=False)
            X_all_tr = torch.FloatTensor(
                processor.scaler.fit_transform(X_all_tr)).to(DEVICE)
            X_all_te = torch.FloatTensor(
                processor.scaler.transform(X_all_te)).to(DEVICE)

        y_tr_t    = torch.FloatTensor(y_tr).to(DEVICE)
        y_te_t    = torch.FloatTensor(y_te).to(DEVICE)
        cat_tr_t  = torch.LongTensor(cat_tr).to(DEVICE)
        cat_te_t  = torch.LongTensor(cat_te).to(DEVICE)
        mask_tr_t = torch.BoolTensor(mask_tr).to(DEVICE)
        mask_te_t = torch.BoolTensor(mask_te).to(DEVICE)
        model     = model.to(DEVICE)

        def weighted_mse_loss(pred, target):
            weights = 1.0 / (target * processor.y_log_max + 1.0)
            weights = weights / weights.mean()
            return ((pred - target) ** 2 * weights).mean()

        optimizer        = optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay)
        patience_counter = 0
        best_model       = None
        train_start_time = time.time()

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            if method_name == "Dual-Stream":
                pred = model(X_sales_tr, X_view_tr, cat_tr_t, mask_tr_t)
            elif method_name == "Adaptive":
                pred = model(X_all_tr, cat_tr_t, mask_tr_t)
            else:
                pred = model(X_all_tr, cat_tr_t)
            loss = weighted_mse_loss(pred, y_tr_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if epoch % 5 == 0:
                model.eval()
                with torch.no_grad():
                    if method_name == "Dual-Stream":
                        val_pred = model(X_sales_te, X_view_te,
                                         cat_te_t, mask_te_t)
                    elif method_name == "Adaptive":
                        val_pred = model(X_all_te, cat_te_t, mask_te_t)
                    else:
                        val_pred = model(X_all_te, cat_te_t)
                    val_loss = weighted_mse_loss(val_pred, y_te_t)
                    if val_loss < best_loss:
                        best_loss        = val_loss
                        patience_counter = 0
                        best_model       = copy.deepcopy(model.state_dict())
                    else:
                        patience_counter += 5
                    if patience_counter >= 25:
                        break

        training_time  = time.time() - train_start_time
        peak_memory_mb = get_peak_gpu_memory_mb()

        if best_model is not None:
            model.load_state_dict(best_model)
        model.eval()

        with torch.no_grad():
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_start = time.time()
            if method_name == "Dual-Stream":
                test_pred = model(X_sales_te, X_view_te, cat_te_t, mask_te_t)
            elif method_name == "Adaptive":
                test_pred = model(X_all_te, cat_te_t, mask_te_t)
            else:
                test_pred = model(X_all_te, cat_te_t)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_time = (time.time() - inf_start) / len(y_te) * 1000

        metrics = get_metrics(test_pred.cpu().numpy(), y_log_te, processor)
        return {
            'metrics':                      metrics,
            'train_time_sec':               training_time,
            'peak_memory_mb':               peak_memory_mb,
            'inference_time_ms_per_sample': inf_time}

    except Exception as e:
        if verbose:
            print(f"Training error: {e}")
        return {
            'metrics':                      {'r2': -np.inf, 'mae': np.inf, 'mape': np.inf},
            'train_time_sec':               0.0,
            'peak_memory_mb':               0.0,
            'inference_time_ms_per_sample': 0.0}


def train_script2_model_run(model, processor, train_data, test_data,
                             method_name, seed, label_encoder,
                             n_categories, sales_dim, input_dim,
                             lr=0.002, weight_decay=1e-5,
                             epochs=150, verbose=False):

    set_all_seeds(seed)
    reset_gpu_stats()
    best_loss = float('inf')

    try:
        X_sales_tr, _, X_all_tr, y_tr, y_log_tr, \
            cat_tr, mask_tr = \
            processor.prepare_features_fair(
                train_data, label_encoder, is_train=True)
        X_sales_te, _, X_all_te, y_te, y_log_te, \
            cat_te, mask_te = \
            processor.prepare_features_fair(
                test_data, label_encoder, is_train=False)

        X_sales_tr = torch.FloatTensor(
            processor.sales_scaler.fit_transform(X_sales_tr)).to(DEVICE)
        X_all_tr   = torch.FloatTensor(
            processor.scaler.fit_transform(X_all_tr)).to(DEVICE)
        X_sales_te = torch.FloatTensor(
            processor.sales_scaler.transform(X_sales_te)).to(DEVICE)
        X_all_te   = torch.FloatTensor(
            processor.scaler.transform(X_all_te)).to(DEVICE)

        y_tr_t    = torch.FloatTensor(y_tr).to(DEVICE)
        y_te_t    = torch.FloatTensor(y_te).to(DEVICE)
        cat_tr_t  = torch.LongTensor(cat_tr).to(DEVICE)
        cat_te_t  = torch.LongTensor(cat_te).to(DEVICE)
        mask_tr_t = torch.BoolTensor(mask_tr).to(DEVICE)
        mask_te_t = torch.BoolTensor(mask_te).to(DEVICE)

        model = model.to(DEVICE)

        def weighted_mse_loss(pred, target):
            weights = 1.0 / (target * processor.y_log_max + 1.0)
            weights = weights / weights.mean()
            return ((pred - target) ** 2 * weights).mean()

        optimizer        = optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay)
        patience_counter = 0
        best_model_state = None
        train_start_time = time.time()

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            pred = model(X_sales_tr, X_all_tr, cat_tr_t, mask_tr_t)
            loss = weighted_mse_loss(pred, y_tr_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if epoch % 5 == 0:
                model.eval()
                with torch.no_grad():
                    val_pred = model(X_sales_te, X_all_te,
                                     cat_te_t, mask_te_t)
                    val_loss = weighted_mse_loss(val_pred, y_te_t)
                    if val_loss < best_loss:
                        best_loss        = val_loss
                        patience_counter = 0
                        best_model_state = copy.deepcopy(model.state_dict())
                    else:
                        patience_counter += 5
                    if patience_counter >= 25:
                        break

        training_time  = time.time() - train_start_time
        peak_memory_mb = get_peak_gpu_memory_mb()

        if best_model_state is not None:
            model.load_state_dict(best_model_state)
        model.eval()

        with torch.no_grad():
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_start = time.time()
            test_pred = model(X_sales_te, X_all_te, cat_te_t, mask_te_t)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_time = (time.time() - inf_start) / len(y_te) * 1000

        metrics = get_metrics(test_pred.cpu().numpy(), y_log_te, processor)
        return {
            'metrics':                      metrics,
            'train_time_sec':               training_time,
            'peak_memory_mb':               peak_memory_mb,
            'inference_time_ms_per_sample': inf_time}

    except Exception as e:
        if verbose:
            print(f"Training error: {e}")
        return {
            'metrics':                      {'r2': -np.inf, 'mae': np.inf, 'mape': np.inf},
            'train_time_sec':               0.0,
            'peak_memory_mb':               0.0,
            'inference_time_ms_per_sample': 0.0}


def train_xgboost_native(train_data, test_data, processor,
                          seed, label_encoder, xgb_params):
    np.random.seed(seed)
    X_all_tr, y_log_tr, cat_tr, _ = \
        processor.prepare_features_xgb_native(
            train_data, label_encoder, is_train=True)
    X_all_te, y_log_te, cat_te, _ = \
        processor.prepare_features_xgb_native(
            test_data, label_encoder, is_train=False)

    X_train = np.column_stack([X_all_tr, cat_tr])
    X_test  = np.column_stack([X_all_te, cat_te])

    scaler     = StandardScaler()
    sales_cols = list(range(N_SALES_FEATURES))
    X_train[:, sales_cols] = scaler.fit_transform(X_train[:, sales_cols])
    X_test[:, sales_cols]  = scaler.transform(X_test[:, sales_cols])

    model = xgb.XGBRegressor(**xgb_params, random_state=seed,
                               n_jobs=-1, verbosity=0,
                               tree_method='hist')
    t0 = time.time()
    model.fit(X_train, y_log_tr)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred_log = model.predict(X_test)
    inf_time   = (time.time() - t1) / len(y_log_te) * 1000

    pred_orig = np.maximum(np.expm1(y_pred_log), 0.1)
    true_orig = np.maximum(np.expm1(y_log_te),   0.1)
    return {
        'metrics': {
            'r2':   r2_score(true_orig, pred_orig),
            'mae':  mean_absolute_error(true_orig, pred_orig),
            'mape': np.median(np.abs(
                (true_orig - pred_orig) /
                np.maximum(true_orig, 1))) * 100},
        'train_time_sec':               train_time,
        'peak_memory_mb':               0.0,
        'inference_time_ms_per_sample': inf_time,
        'y_pred_log': y_pred_log,
        'y_true_log': y_log_te}


def train_saint(train_data, test_data, processor,
                seed, label_encoder,
                d_model=32, nhead=2, n_layers=2,
                lr=0.001, weight_decay=1e-4,
                dropout=0.1, epochs=150,
                n_categories=18, verbose=False):

    set_all_seeds(seed)
    reset_gpu_stats()
    best_loss = float('inf')

    try:
        _, _, X_all_tr, y_tr, y_log_tr, cat_tr, _ = \
            processor.prepare_features_fair(
                train_data, label_encoder, is_train=True)
        _, _, X_all_te, y_te, y_log_te, cat_te, _ = \
            processor.prepare_features_fair(
                test_data, label_encoder, is_train=False)

        X_all_tr = torch.FloatTensor(
            processor.scaler.fit_transform(X_all_tr)).to(DEVICE)
        X_all_te = torch.FloatTensor(
            processor.scaler.transform(X_all_te)).to(DEVICE)

        y_tr_t   = torch.FloatTensor(y_tr).to(DEVICE)
        y_te_t   = torch.FloatTensor(y_te).to(DEVICE)
        cat_tr_t = torch.LongTensor(cat_tr).to(DEVICE)
        cat_te_t = torch.LongTensor(cat_te).to(DEVICE)

        model = SAINTModel(
            n_cont_features=N_TOTAL_FEATURES,
            n_categories=n_categories,
            d_model=d_model, nhead=nhead,
            n_layers=n_layers, dropout=dropout).to(DEVICE)

        def weighted_mse_loss(pred, target):
            weights = 1.0 / (target * processor.y_log_max + 1.0)
            weights = weights / weights.mean()
            return ((pred - target) ** 2 * weights).mean()

        optimizer        = optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay)
        patience_counter = 0
        best_model       = None
        train_start_time = time.time()

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            pred = model(X_all_tr, cat_tr_t)
            loss = weighted_mse_loss(pred, y_tr_t)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            if epoch % 5 == 0:
                model.eval()
                with torch.no_grad():
                    val_pred = model(X_all_te, cat_te_t)
                    val_loss = weighted_mse_loss(val_pred, y_te_t)
                    if val_loss < best_loss:
                        best_loss        = val_loss
                        patience_counter = 0
                        best_model       = copy.deepcopy(model.state_dict())
                    else:
                        patience_counter += 5
                    if patience_counter >= 25:
                        break

        training_time  = time.time() - train_start_time
        peak_memory_mb = get_peak_gpu_memory_mb()

        if best_model is not None:
            model.load_state_dict(best_model)
        model.eval()

        with torch.no_grad():
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_start = time.time()
            test_pred = model(X_all_te, cat_te_t)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            inf_time = (time.time() - inf_start) / len(y_te) * 1000

        metrics = get_metrics(test_pred.cpu().numpy(), y_log_te, processor)
        return {
            'metrics':                      metrics,
            'train_time_sec':               training_time,
            'peak_memory_mb':               peak_memory_mb,
            'inference_time_ms_per_sample': inf_time,
            'y_pred_log':                   test_pred.cpu().numpy(),
            'y_true_log':                   y_log_te}

    except Exception as e:
        if verbose:
            print(f"SAINT training error: {e}")
        return {
            'metrics':                      {'r2': -np.inf, 'mae': np.inf, 'mape': np.inf},
            'train_time_sec':               0.0,
            'peak_memory_mb':               0.0,
            'inference_time_ms_per_sample': 0.0}


print("Training functions loaded.")
print("  train_script1_model_run — Baseline, Dual-Stream, Adaptive")
print("  train_script2_model_run — GatedFusion")
print("  train_xgboost_zero      — XGBoost (zero-fill)")
print("  train_xgboost_native    — XGBoost (NaN-native)")
print("  train_saint             — SAINT")

Training functions loaded.
  train_script1_model_run — Baseline, Dual-Stream, Adaptive
  train_script2_model_run — GatedFusion
  train_xgboost_zero      — XGBoost (zero-fill)
  train_xgboost_native    — XGBoost (NaN-native)
  train_saint             — SAINT


In [9]:
#Phase 1

import json
from scipy.stats import wilcoxon as _wilcoxon

#with open('results/hpo_optimal_params.json', 'r') as f:
#    optimal_params_from_hpo = json.load(f)
#print("Loaded optimal params:")
#print(json.dumps(optimal_params_from_hpo, indent=2))

optimal_params_from_hpo = {
  "Baseline": {
      "lr": 0.005,
      "d_model": 32,
      "dropout": 0.1,
      "weight_decay": 1e-05
    },
    "Dual-Stream": {
      "lr": 0.005,
      "d_model": 64,
      "dropout": 0.1,
      "weight_decay": 1e-05
    },
    "Adaptive": {
      "lr": 0.005,
      "d_model": 32,
      "dropout": 0.1,
      "weight_decay": 1e-05,
      "alpha": 0.3
    },
    "GatedFusion": {
      "lr": 0.003,
      "d_model": 64,
      "dropout": 0.1,
      "weight_decay": 1e-05,
      "category_emb_dim": 16
    },
    "XGBoost": {
      "n_estimators": 200,
      "max_depth": 6,
      "learning_rate": 0.05,
      "subsample": 0.8,
      "colsample_bytree": 0.7
    },
    "SAINT": {
        "d_model": 32,
        "nhead": 2,
        "n_layers": 2,
        "lr": 0.001,
        "weight_decay": 1e-4,
        "dropout": 0.1,
        "epochs": 150
    }
}

def compute_significance_kfold(results_dict):
    print(f"\n{'='*60}")
    print("PHASE 1: STATISTICAL SIGNIFICANCE (R2)")
    print(f"{'='*60}")
    methods = list(results_dict.keys())
    if not methods:
        return None, {}

    best_r2_method = max(methods, key=lambda m: results_dict[m]['r2_mean'])
    print(f"\nBest model: {best_r2_method}")
    print(f"R2 = {results_dict[best_r2_method]['r2_mean']:.4f}"
          f" +/- {results_dict[best_r2_method]['r2_std']:.4f}")
    print(f"\n{'Method':<25} {'Mean R2':<12} {'t-test p':<12} {'Wilcoxon p':<14} {'Sig'}")
    print("-" * 72)

    significance_markers = {}
    best_scores = results_dict[best_r2_method]['r2_values']

    for method in methods:
        if method == best_r2_method:
            print(f"{method:<25} {results_dict[method]['r2_mean']:.4f}"
                  f"       —            —              (reference)")
            significance_markers[method] = ""
            continue

        method_scores = results_dict[method]['r2_values']
        if len(best_scores) != len(method_scores):
            continue

        t_stat, p_ttest = stats.ttest_rel(best_scores, method_scores)

        try:
            w_stat, p_wilcoxon = _wilcoxon(
                best_scores, method_scores, alternative='greater')
            w_str = f"{p_wilcoxon:.4f}"
        except Exception:
            w_str = "N/A"

        if   p_ttest < 0.001: marker = "***"
        elif p_ttest < 0.01:  marker = "**"
        elif p_ttest < 0.05:  marker = "*"
        else:                  marker = "ns"

        significance_markers[method] = marker if marker != "ns" else ""
        print(f"{method:<25} {results_dict[method]['r2_mean']:.4f}"
              f"       {p_ttest:.4f}       {w_str:<14} {marker}")

    return best_r2_method, significance_markers


def generate_phase1_table(results_dict, significance_markers, K):
    print(f"\n{'='*80}")
    print(f"PHASE 1 (K={K}): RESULTS TABLE")
    print(f"{'='*80}\n")
    print(f"{'Method':<25} {'R2 (mean+/-std)':<22} {'MAPE (%)':<18} "
          f"{'MAE':<18} {'Train (s)':<15} {'Memory (MB)':<15} {'Inference (ms)'}")
    print("-" * 120)

    for method in ['Baseline', 'Dual-Stream', 'Adaptive', 'GatedFusion']:
        if method not in results_dict:
            continue
        r   = results_dict[method]
        sig = significance_markers.get(method, '')
        print(f"{method:<25} "
              f"{r['r2_mean']:.4f} +/- {r['r2_std']:.4f}{sig:<3}  "
              f"{r['mape_mean']:.1f} +/- {r['mape_std']:.1f}      "
              f"{r['mae_mean']:.2f} +/- {r['mae_std']:.2f}   "
              f"{r['train_mean']:.2f} +/- {r['train_std']:.2f}   "
              f"{r['peak_mean']:.2f} +/- {r['peak_std']:.2f}   "
              f"{r['inference_mean']:.4f} +/- {r['inference_std']:.4f}")

    print(f"\nSignificance (* p<0.05, ** p<0.01, *** p<0.001) "
          f"vs best model — paired t-test, K={K}")


def run_phase1_tournament(K, full_data, label_encoder,
                           model_definitions, model_dims,
                           n_categories, stratify_groups,
                           optimal_params):
    print("\n" + "="*80)
    print(f"PHASE 1: TRANSFORMER TOURNAMENT (K={K})")
    print("="*80)

    sales_dim, view_dim, input_dim = model_dims
    kfold = StratifiedKFold(n_splits=K, shuffle=True,
                             random_state=CV_RANDOM_STATE)

    fold_results_agg = {name: [] for name in model_definitions}
    all_predictions  = {name: {'y_true': [], 'y_pred': []}
                        for name in fold_results_agg}

    for fold, (train_idx, test_idx) in enumerate(
            kfold.split(full_data, stratify_groups), 1):
        print(f"\n--- Fold {fold}/{K} ---")
        train_data = full_data.iloc[train_idx]
        test_data  = full_data.iloc[test_idx]

        for model_name in fold_results_agg:
            print(f"   Training: {model_name}")
            params       = optimal_params[model_name]
            model_kwargs = copy.deepcopy(
                model_definitions[model_name]['base_kwargs'])
            model_kwargs['d_model'] = params.get('d_model', 32)
            model_kwargs['dropout'] = params.get('dropout', 0.1)

            if model_name == 'Adaptive' and 'alpha' in params:
                model_kwargs['alpha'] = params['alpha']
            if model_name == 'GatedFusion' and 'category_emb_dim' in params:
                model_kwargs['category_emb_dim'] = params['category_emb_dim']

            model_instance = model_definitions[model_name]['class'](**model_kwargs)
            processor      = ImprovedDataProcessor()

            try:
                if model_name == 'GatedFusion':
                    run_metrics = model_definitions[model_name]['train_fn'](
                        model_instance, processor,
                        train_data, test_data, model_name,
                        CV_RANDOM_STATE + fold,
                        label_encoder, n_categories,
                        sales_dim, input_dim,
                        lr=params['lr'],
                        weight_decay=params['weight_decay'])
                else:
                    run_metrics = model_definitions[model_name]['train_fn'](
                        model_instance, processor,
                        train_data, test_data, model_name,
                        CV_RANDOM_STATE + fold,
                        label_encoder,
                        lr=params['lr'],
                        weight_decay=params['weight_decay'])

                fold_results_agg[model_name].append(run_metrics)
                print(f"      R2={run_metrics['metrics']['r2']:.4f} "
                      f"({run_metrics['train_time_sec']:.1f}s)")

            except Exception as e:
                print(f"      ERROR: {e}")
                fold_results_agg[model_name].append(None)

    print("\n" + "="*80)
    print(f"PHASE 1 (K={K}) SUMMARY")
    print("="*80)

    summary_stats = {}
    for model_name, fold_runs in fold_results_agg.items():
        valid_runs = [r for r in fold_runs
                      if r is not None and r['metrics']['r2'] > -np.inf]
        if not valid_runs:
            print(f"\n{model_name}: no successful folds")
            continue

        summary = {'method': model_name}
        for key in valid_runs[0]['metrics']:
            if key == 'val_loss':
                continue
            summary[f'{key}_values'] = [r['metrics'][key] for r in valid_runs]
        summary['train_values']     = [r['train_time_sec'] for r in valid_runs]
        summary['peak_values']      = [r['peak_memory_mb'] for r in valid_runs]
        summary['inference_values'] = [
            r['inference_time_ms_per_sample'] * 1000 for r in valid_runs]

        for key in list(summary.keys()):
            if key.endswith('_values'):
                vals = summary[key]
                base = key.replace('_values', '')
                summary[f'{base}_mean'] = np.mean(vals)
                summary[f'{base}_std']  = np.std(vals, ddof=1)

        summary_stats[model_name] = summary
        print(f"\n{model_name} ({len(valid_runs)}/{K} folds)")
        print(f"  R2:   {summary['r2_mean']:.4f} +/- {summary['r2_std']:.4f}")
        print(f"  MAE:  {summary['mae_mean']:.2f} +/- {summary['mae_std']:.2f}")
        print(f"  MAPE: {summary['mape_mean']:.1f} +/- {summary['mape_std']:.1f}%")

    best_transformer, sig_markers = compute_significance_kfold(summary_stats)
    generate_phase1_table(summary_stats, sig_markers, K)

    return best_transformer, summary_stats


full_data     = ImprovedDataProcessor().load_full_dataset()
label_encoder = LabelEncoder()
label_encoder.fit(
    full_data['sales_main_category'].fillna('Unknown').astype(str).str.strip())

stratify_groups = ImprovedDataProcessor().get_stratify_groups(full_data)
n_categories    = len(label_encoder.classes_)
sales_dim       = N_SALES_FEATURES
view_dim        = N_VIEW_FEATURES
input_dim       = N_TOTAL_FEATURES
model_dims      = (sales_dim, view_dim, input_dim)

model_definitions = {
    'Baseline': {
        'class':       BaselineTransformer,
        'base_kwargs': {'input_dim': input_dim, 'n_categories': n_categories, 'nhead': 4},
        'train_fn':    train_script1_model_run},
    'Dual-Stream': {
        'class':       DualStreamTransformer,
        'base_kwargs': {'sales_dim': sales_dim, 'view_dim': view_dim,
                        'n_categories': n_categories, 'nhead': 4},
        'train_fn':    train_script1_model_run},
    'Adaptive': {
        'class':       AdaptiveWeightingTransformer,
        'base_kwargs': {'input_dim': input_dim, 'n_categories': n_categories, 'nhead': 4},
        'train_fn':    train_script1_model_run},
    'GatedFusion': {
        'class':       GatedFusionTransformer,
        'base_kwargs': {'sales_dim': sales_dim, 'full_dim': input_dim,
                        'n_categories': n_categories, 'nhead': 4, 'num_layers': 1},
        'train_fn':    train_script2_model_run}
}

best_transformer, phase1_results = run_phase1_tournament(
    K_FOLDS, full_data, label_encoder,
    model_definitions, model_dims,
    n_categories, stratify_groups,
    optimal_params_from_hpo)

print(f"\nBest transformer: {best_transformer}")


PHASE 1: TRANSFORMER TOURNAMENT (K=5)

--- Fold 1/5 ---
   Training: Baseline
      R2=0.8161 (0.6s)
   Training: Dual-Stream
      R2=0.8876 (0.6s)
   Training: Adaptive
      R2=0.8654 (0.6s)
   Training: GatedFusion
      R2=0.9214 (1.5s)

--- Fold 2/5 ---
   Training: Baseline
      R2=0.7919 (0.6s)
   Training: Dual-Stream
      R2=0.6595 (0.6s)
   Training: Adaptive
      R2=0.6288 (0.6s)
   Training: GatedFusion
      R2=0.8630 (1.4s)

--- Fold 3/5 ---
   Training: Baseline
      R2=0.7045 (0.6s)
   Training: Dual-Stream
      R2=0.8666 (0.6s)
   Training: Adaptive
      R2=0.8106 (0.6s)
   Training: GatedFusion
      R2=0.9128 (1.5s)

--- Fold 4/5 ---
   Training: Baseline
      R2=0.7853 (0.6s)
   Training: Dual-Stream
      R2=0.8321 (0.6s)
   Training: Adaptive
      R2=0.6250 (0.6s)
   Training: GatedFusion
      R2=0.9360 (1.6s)

--- Fold 5/5 ---
   Training: Baseline
      R2=0.7708 (0.6s)
   Training: Dual-Stream
      R2=0.8113 (0.6s)
   Training: Adaptive
      R2=0.8

In [12]:
from scipy.stats import wilcoxon as wilcoxon_test


def evaluate_subset(model_name, model_instance, processor,
                    train_data, subset_data, label_encoder,
                    n_categories, sales_dim, input_dim,
                    params, seed,
                    is_saint=False, is_xgb_zero=False, is_xgb_native=False):
    """Trains on full train_data, evaluates on subset_data. Returns R2."""
    if is_xgb_native:
        return train_xgboost_native(
            train_data, subset_data, processor,
            seed, label_encoder, xgb_params=params)['metrics']['r2']
    elif is_saint:
        return train_saint(
            train_data, subset_data, processor,
            seed, label_encoder,
            d_model=params['d_model'], nhead=params['nhead'],
            n_layers=params['n_layers'], lr=params['lr'],
            weight_decay=params['weight_decay'],
            dropout=params['dropout'],
            epochs=params.get('epochs', 150),
            n_categories=n_categories)['metrics']['r2']
    else:
        return train_script2_model_run(
            model_instance, processor,
            train_data, subset_data, 'GatedFusion', seed,
            label_encoder, n_categories, sales_dim, input_dim,
            lr=params['lr'],
            weight_decay=params['weight_decay'])['metrics']['r2']


def run_phase2_extended(K, full_data, label_encoder,
                         model_dims, n_categories,
                         stratify_groups, optimal_params):

    print("\n" + "="*80)
    print(f"PHASE 2: EXTENDED COMPARISON (K={K})")
    print("  GatedFusion | XGBoost-Native | SAINT")
    print("="*80)

    sales_dim, view_dim, input_dim = model_dims
    kfold = StratifiedKFold(n_splits=K, shuffle=True,
                             random_state=CV_RANDOM_STATE)

    params_gf  = optimal_params['GatedFusion']
    params_xgb = optimal_params['XGBoost']
    params_saint = optimal_params.get('SAINT', {
        'd_model': 32, 'nhead': 2, 'n_layers': 2,
        'lr': 0.001, 'weight_decay': 1e-4,
        'dropout': 0.1, 'epochs': 150})

    results_fusion = []; results_xgb_z = []
    results_xgb_n  = []; results_saint = []
    subset_fusion  = []; subset_xgb_z  = []
    subset_xgb_n   = []; subset_saint  = []

    for fold, (train_idx, test_idx) in enumerate(
            kfold.split(full_data, stratify_groups), 1):

        print(f"\n{'─'*60}")
        print(f"Fold {fold}/{K}")
        print(f"{'─'*60}")

        train_data = full_data.iloc[train_idx]
        test_data  = full_data.iloc[test_idx]

        has_view      = test_data['view_count'].fillna(0) > 0
        complete_data = test_data[has_view].copy()
        missing_data  = test_data[~has_view].copy()
        do_subset     = len(complete_data) > 0 and len(missing_data) > 0

        print(f"  Test: {len(test_data)} total | "
              f"{len(complete_data)} complete | {len(missing_data)} missing")

        # ── 1. GATED FUSION ──────────────────────────────────────────
        print("\n  [1/3] GatedFusion...")
        model_kwargs = {
            'sales_dim':        sales_dim,
            'full_dim':         input_dim,
            'n_categories':     n_categories,
            'nhead':            4,
            'num_layers':       1,
            'd_model':          params_gf.get('d_model', 64),
            'dropout':          params_gf.get('dropout', 0.1),
            'category_emb_dim': params_gf.get('category_emb_dim', 8)}

        proc_f  = ImprovedDataProcessor()
        model_f = GatedFusionTransformer(**model_kwargs)
        res_f   = train_script2_model_run(
            model_f, proc_f, train_data, test_data,
            'GatedFusion', CV_RANDOM_STATE + fold,
            label_encoder, n_categories, sales_dim, input_dim,
            lr=params_gf['lr'], weight_decay=params_gf['weight_decay'])
        results_fusion.append(res_f)
        print(f"     R2: {res_f['metrics']['r2']:.4f}")

        if do_subset:
            proc_fc  = ImprovedDataProcessor()
            model_fc = GatedFusionTransformer(**model_kwargs)
            r2_fc    = evaluate_subset(
                'GatedFusion', model_fc, proc_fc,
                train_data, complete_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_gf, CV_RANDOM_STATE + fold)

            proc_fm  = ImprovedDataProcessor()
            model_fm = GatedFusionTransformer(**model_kwargs)
            r2_fm    = evaluate_subset(
                'GatedFusion', model_fm, proc_fm,
                train_data, missing_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_gf, CV_RANDOM_STATE + fold)

            subset_fusion.append({'complete': r2_fc, 'missing': r2_fm})
            print(f"     Complete: {r2_fc:.4f} | Missing: {r2_fm:.4f}")
        else:
            subset_fusion.append(None)

        # ── 2. XGBOOST-NATIVE ────────────────────────────────────────
        print("\n  [2/3] XGBoost-Native...")
        proc_xn = ImprovedDataProcessor()
        res_xn  = train_xgboost_native(
            train_data, test_data, proc_xn,
            CV_RANDOM_STATE + fold, label_encoder, xgb_params=params_xgb)
        results_xgb_n.append(res_xn)
        print(f"     R2: {res_xn['metrics']['r2']:.4f}")

        if do_subset:
            proc_xnc = ImprovedDataProcessor()
            r2_xnc   = evaluate_subset(
                'XGBoost-Native', None, proc_xnc,
                train_data, complete_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_xgb, CV_RANDOM_STATE + fold, is_xgb_native=True)
            proc_xnm = ImprovedDataProcessor()
            r2_xnm   = evaluate_subset(
                'XGBoost-Native', None, proc_xnm,
                train_data, missing_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_xgb, CV_RANDOM_STATE + fold, is_xgb_native=True)
            subset_xgb_n.append({'complete': r2_xnc, 'missing': r2_xnm})
            print(f"     Complete: {r2_xnc:.4f} | Missing: {r2_xnm:.4f}")
        else:
            subset_xgb_n.append(None)

        # ── 3. SAINT ─────────────────────────────────────────────────
        print("\n  [3/3] SAINT...")
        proc_s = ImprovedDataProcessor()
        res_s  = train_saint(
            train_data, test_data, proc_s,
            CV_RANDOM_STATE + fold, label_encoder,
            d_model=params_saint['d_model'],
            nhead=params_saint['nhead'],
            n_layers=params_saint['n_layers'],
            lr=params_saint['lr'],
            weight_decay=params_saint['weight_decay'],
            dropout=params_saint['dropout'],
            epochs=params_saint.get('epochs', 150),
            n_categories=n_categories)
        results_saint.append(res_s)
        print(f"     R2: {res_s['metrics']['r2']:.4f}")

        if do_subset:
            proc_sc = ImprovedDataProcessor()
            r2_sc   = evaluate_subset(
                'SAINT', None, proc_sc,
                train_data, complete_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_saint, CV_RANDOM_STATE + fold, is_saint=True)
            proc_sm = ImprovedDataProcessor()
            r2_sm   = evaluate_subset(
                'SAINT', None, proc_sm,
                train_data, missing_data, label_encoder,
                n_categories, sales_dim, input_dim,
                params_saint, CV_RANDOM_STATE + fold, is_saint=True)
            subset_saint.append({'complete': r2_sc, 'missing': r2_sm})
            print(f"     Complete: {r2_sc:.4f} | Missing: {r2_sm:.4f}")
        else:
            subset_saint.append(None)

    # ── SUMMARY FUNCTIONS ─────────────────────────────────────────────────
    def summarize(results, name):
        r2s = [r['metrics']['r2']   for r in results]
        return {
            'name':      name,
            'r2_mean':   np.mean(r2s),
            'r2_std':    np.std(r2s, ddof=1),
            'r2_values': r2s,
            'mae_mean':  np.mean([r['metrics']['mae']  for r in results]),
            'mae_std':   np.std( [r['metrics']['mae']  for r in results], ddof=1),
            'mape_mean': np.mean([r['metrics']['mape'] for r in results]),
            'mape_std':  np.std( [r['metrics']['mape'] for r in results], ddof=1),
            'time_mean': np.mean([r['train_time_sec']  for r in results]),
            'time_std':  np.std( [r['train_time_sec']  for r in results], ddof=1)}

    def subset_summary(subset_list, name):
        valid = [s for s in subset_list if s is not None]
        if not valid:
            return None
        comp = [s['complete'] for s in valid]
        miss = [s['missing']  for s in valid]
        return {
            'name':            name,
            'complete_mean':   np.mean(comp),
            'complete_std':    np.std(comp, ddof=1),
            'complete_values': comp,
            'missing_mean':    np.mean(miss),
            'missing_std':     np.std(miss, ddof=1),
            'missing_values':  miss}

    s_fusion = summarize(results_fusion, 'GatedFusion')
    s_xgb_n  = summarize(results_xgb_n,  'XGBoost-Native')
    s_saint  = summarize(results_saint,  'SAINT')

    ss_fusion = subset_summary(subset_fusion, 'GatedFusion')
    ss_xgb_n  = subset_summary(subset_xgb_n,  'XGBoost-Native')
    ss_saint  = subset_summary(subset_saint,  'SAINT')

    all_summaries = [s_fusion, s_xgb_n, s_saint]
    all_ss        = [ss_fusion, ss_xgb_n, ss_saint]

    # ── PHASE 2A ──────────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("PHASE 2A: GENERAL PERFORMANCE (K=5)")
    print("="*80)
    print(f"\n{'Method':<20} {'R2 (mean±std)':<22} {'MAE':<18} {'MAPE(%)':<12} {'Time(s)'}")
    print("─" * 82)
    for s in all_summaries:
        print(f"{s['name']:<20} "
              f"{s['r2_mean']:.4f} ± {s['r2_std']:.4f}   "
              f"{s['mae_mean']:.2f} ± {s['mae_std']:.2f}   "
              f"{s['mape_mean']:.1f} ± {s['mape_std']:.1f}   "
              f"{s['time_mean']:.2f} ± {s['time_std']:.2f}")

    # ── PHASE 2B ──────────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("PHASE 2B: COMPLETE vs MISSING SUBSET ANALYSIS")
    print("="*80)
    print(f"\n{'Method':<20} {'Complete R2':<24} {'Missing R2':<24} {'Diff (M-C)'}")
    print("─" * 75)
    for ss in all_ss:
        if ss is None:
            continue
        diff = ss['missing_mean'] - ss['complete_mean']
        print(f"{ss['name']:<20} "
              f"{ss['complete_mean']:.4f} ± {ss['complete_std']:.4f}   "
              f"{ss['missing_mean']:.4f} ± {ss['missing_std']:.4f}   "
              f"{diff:+.4f}")

    # ── PHASE 2C ──────────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("PHASE 2C: STATISTICAL TESTS")
    print("  Interaction effect: GatedFusion advantage on missing vs complete data")
    print("="*80)

    def run_statistical_tests(ss_model, model_name):
        if ss_model is None or ss_fusion is None:
            print(f"\n  {model_name}: insufficient data")
            return

        n = min(len(ss_fusion['complete_values']),
                len(ss_model['complete_values']))

        gap_complete = (np.array(ss_fusion['complete_values'][:n]) -
                        np.array(ss_model['complete_values'][:n]))
        gap_missing  = (np.array(ss_fusion['missing_values'][:n]) -
                        np.array(ss_model['missing_values'][:n]))
        interaction  = gap_missing - gap_complete

        print(f"\n  GatedFusion vs {model_name}:")
        print(f"  {'Fold':<6} {'Gap(Complete)':<16} {'Gap(Missing)':<16} {'Interaction'}")
        print(f"  {'─'*52}")
        for i in range(n):
            print(f"  {i+1:<6} {gap_complete[i]:+.4f}         "
                  f"{gap_missing[i]:+.4f}         "
                  f"{interaction[i]:+.4f}")

        mean_int = np.mean(interaction)
        print(f"\n  Mean interaction : {mean_int:+.4f}")
        print(f"  Fusion favours missing more than complete: "
              f"{'YES' if mean_int > 0 else 'NO'}")

        t_stat, p_ttest = stats.ttest_1samp(interaction, 0)
        sig_t = ('***' if p_ttest < 0.001 else '**' if p_ttest < 0.01
                 else '*' if p_ttest < 0.05 else 'ns')
        print(f"\n  t-test  : t={t_stat:.4f}, p={p_ttest:.4f} ({sig_t})")

        try:
            w_stat, p_wilcoxon = wilcoxon_test(interaction, alternative='greater')
            sig_w = ('***' if p_wilcoxon < 0.001 else '**' if p_wilcoxon < 0.01
                     else '*' if p_wilcoxon < 0.05 else 'ns')
            print(f"  Wilcoxon: W={w_stat:.4f}, p={p_wilcoxon:.4f} ({sig_w})")
        except Exception as e:
            print(f"  Wilcoxon: could not compute ({e})")

        print(f"\n  Win rate (GatedFusion > {model_name}):")
        print(f"    Complete: {np.sum(gap_complete > 0)}/{n} folds")
        print(f"    Missing : {np.sum(gap_missing  > 0)}/{n} folds")

    run_statistical_tests(ss_xgb_n,  'XGBoost-Native')
    run_statistical_tests(ss_saint,  'SAINT')

    # ── FINAL TABLE ───────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("PHASE 2: FINAL SUMMARY")
    print("="*80)
    print(f"\n{'Model':<20} {'Overall R2':<22} {'Complete R2':<22} {'Missing R2'}")
    print("─" * 86)
    for s, ss in zip(all_summaries, all_ss):
        if ss is None:
            print(f"{s['name']:<20} {s['r2_mean']:.4f}±{s['r2_std']:.4f}   "
                  f"{'N/A':<22} {'N/A'}")
        else:
            print(f"{s['name']:<20} {s['r2_mean']:.4f}±{s['r2_std']:.4f}   "
                  f"{ss['complete_mean']:.4f}±{ss['complete_std']:.4f}      "
                  f"{ss['missing_mean']:.4f}±{ss['missing_std']:.4f}")

    # ── SAVE ─────────────────────────────────────────────────────────────
    Path('results').mkdir(exist_ok=True)
    phase2_results = {
        'general': {
            s['name']: {'r2_mean': s['r2_mean'],
                        'r2_std':  s['r2_std'],
                        'r2_values': s['r2_values']}
            for s in all_summaries},
        'subsets': {
            ss['name']: {'complete': ss['complete_values'],
                         'missing':  ss['missing_values']}
            for ss in all_ss if ss is not None}}

    with open('results/phase2_extended_results.json', 'w') as f:
        json.dump(phase2_results, f, indent=2)
    print("\nSaved to results/phase2_extended_results.json")

    return phase2_results


# Run
phase2_results = run_phase2_extended(
    K_FOLDS, full_data, label_encoder,
    model_dims, n_categories,
    stratify_groups, optimal_params_from_hpo)



PHASE 2: EXTENDED COMPARISON (K=5)
  GatedFusion | XGBoost-Native | SAINT

────────────────────────────────────────────────────────────
Fold 1/5
────────────────────────────────────────────────────────────
  Test: 1346 total | 655 complete | 691 missing

  [1/3] GatedFusion...
     R2: 0.9284
     Complete: 0.9226 | Missing: 0.9372

  [2/3] XGBoost-Native...
     R2: 0.9326
     Complete: 0.9294 | Missing: 0.9376

  [3/3] SAINT...
     R2: 0.7192
     Complete: 0.6897 | Missing: 0.7752

────────────────────────────────────────────────────────────
Fold 2/5
────────────────────────────────────────────────────────────
  Test: 1346 total | 614 complete | 732 missing

  [1/3] GatedFusion...
     R2: 0.8664
     Complete: 0.8516 | Missing: 0.8899

  [2/3] XGBoost-Native...
     R2: 0.9319
     Complete: 0.9307 | Missing: 0.9321

  [3/3] SAINT...
     R2: 0.7497
     Complete: 0.7184 | Missing: 0.8371

────────────────────────────────────────────────────────────
Fold 3/5
────────────────────